# 🔧 Fix & Upload: Merge_Docs_Pro_Folder → Beyond-Threshold-Monitoring

This notebook:
1. Mounts Drive
2. **Fixes all `.ipynb` files directly on Google Drive** (permanent source fix)
3. Clones or initialises the GitHub repo
4. Copies all files from `Merge_Docs_Pro_Folder` into the repo
5. Verifies every notebook passes GitHub rendering requirements
6. Commits and pushes to GitHub

---
**No need to delete/recreate the repo.** If it already exists the push will simply update it.

## ⚙️ Configuration

In [ ]:
# ── DO NOT CHANGE THESE ────────────────────────────────────────────────────────
GITHUB_USERNAME       = "o-brown7025"
GITHUB_REPO           = "Beyond-Threshold-Monitoring"
GIT_EMAIL             = "o.brown7025@student.nu.edu"
SECRET_NAME           = "Beyond-Threshold-Monitoring"
DRIVE_SOURCE_FOLDER   = "/content/drive/MyDrive/Merge_Docs_Pro_Folder"
LOCAL_REPO            = f"/content/{GITHUB_REPO}"
# ──────────────────────────────────────────────────────────────────────────────

## Step 1 — Mount Drive & configure Git

In [ ]:
from google.colab import drive, userdata
import subprocess, os, shutil, json, pathlib, uuid

drive.mount('/content/drive')

GITHUB_TOKEN = userdata.get(SECRET_NAME)

subprocess.run(["git", "config", "--global", "user.name",  GITHUB_USERNAME], check=True)
subprocess.run(["git", "config", "--global", "user.email", GIT_EMAIL],       check=True)

# Verify the source folder exists
src = pathlib.Path(DRIVE_SOURCE_FOLDER)
if src.exists():
    all_files = list(src.rglob("*"))
    nb_files  = list(src.rglob("*.ipynb"))
    print(f"✅ Drive mounted")
    print(f"✅ Source folder found: {DRIVE_SOURCE_FOLDER}")
    print(f"   Total files : {len(all_files)}")
    print(f"   Notebooks   : {len(nb_files)}")
else:
    print(f"❌ Source folder NOT found: {DRIVE_SOURCE_FOLDER}")
    print("   Check the path above — it must match exactly (case-sensitive).")

## Step 2 — Fix all notebooks directly on Google Drive
This permanently repairs the source files so every future upload is already correct.

In [ ]:
KERNELSPEC = {
    "display_name": "Python 3",
    "language": "python",
    "name": "python3"
}
LANGUAGE_INFO = {
    "codemirror_mode": {"name": "ipython", "version": 3},
    "file_extension": ".py",
    "mimetype": "text/x-python",
    "name": "python",
    "pygments_lexer": "ipython3",
    "version": "3.10.12"
}

drive_notebooks = list(pathlib.Path(DRIVE_SOURCE_FOLDER).rglob("*.ipynb"))

if not drive_notebooks:
    print("⚠ No .ipynb files found in the source folder.")
    print("  This folder may contain only non-notebook files — that's fine, continue to Step 3.")
else:
    print(f"Found {len(drive_notebooks)} notebook(s) on Drive — fixing now...\n")

fixed, skipped, failed = 0, 0, 0

for nb_path in drive_notebooks:
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))

        # Fix format version
        nb["nbformat"]       = 4
        nb["nbformat_minor"] = 5

        # Fix metadata
        nb.setdefault("metadata", {})
        nb["metadata"]["kernelspec"]    = KERNELSPEC
        nb["metadata"]["language_info"] = LANGUAGE_INFO

        # Fix cells
        for cell in nb.get("cells", []):
            if "id" not in cell or not cell["id"]:
                cell["id"] = uuid.uuid4().hex[:8]
            # Strip outputs to prevent GitHub size/timeout errors
            if cell.get("cell_type") == "code":
                cell["outputs"]         = []
                cell["execution_count"] = None

        nb_path.write_text(
            json.dumps(nb, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )
        fixed += 1
        rel = nb_path.relative_to(DRIVE_SOURCE_FOLDER)
        print(f"  ✅ Fixed: {rel}")

    except Exception as e:
        failed += 1
        print(f"  ❌ {nb_path.name}: {e}")

print(f"\n🎯 {fixed} fixed on Drive, {failed} failed")

## Step 3 — Clone the GitHub repo (fresh every time)

In [ ]:
# Always start from a clean clone
if os.path.exists(LOCAL_REPO):
    shutil.rmtree(LOCAL_REPO)
    print("🗑  Removed old local clone")

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

result = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, LOCAL_REPO],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f"✅ Cloned existing repo to {LOCAL_REPO}")
else:
    # Repo may not exist yet — initialise a new one
    print("ℹ Repo not found or empty — initialising new local repo...")
    os.makedirs(LOCAL_REPO, exist_ok=True)
    subprocess.run(["git", "init", "-b", "main"], cwd=LOCAL_REPO, check=True)
    subprocess.run(["git", "remote", "add", "origin", REPO_URL], cwd=LOCAL_REPO)
    print("✅ New local repo initialised")
    print("   Make sure you've created the repo on GitHub first at:")
    print(f"   https://github.com/new  →  name it: {GITHUB_REPO}")

## Step 4 — Scan what's in the Drive source folder

In [ ]:
src = pathlib.Path(DRIVE_SOURCE_FOLDER)
all_items = sorted(src.rglob("*"))

print(f"Contents of {DRIVE_SOURCE_FOLDER}:\n")
for item in all_items:
    rel = item.relative_to(src)
    kind = "📁" if item.is_dir() else "📄"
    print(f"  {kind} {rel}")

print(f"\nTotal: {len([x for x in all_items if x.is_file()])} files, "
      f"{len([x for x in all_items if x.is_dir()])} subdirectories")

## Step 5 — Copy all files from Drive into the repo
Preserves the full subfolder structure exactly as-is on Drive.

In [ ]:
src  = pathlib.Path(DRIVE_SOURCE_FOLDER)
dest = pathlib.Path(LOCAL_REPO)

# Large file extensions that will be skipped (GitHub 100MB limit)
SKIP_EXTENSIONS = {".h5", ".hdf5", ".pkl", ".pickle", ".pt", ".pth",
                   ".npy", ".npz", ".zip", ".tar", ".gz", ".rar",
                   ".mp4", ".avi", ".mov", ".wav", ".mp3"}

copied, skipped_large, skipped_git = 0, 0, 0

for src_file in src.rglob("*"):
    if not src_file.is_file():
        continue

    # Skip hidden git internals
    if ".git" in src_file.parts:
        skipped_git += 1
        continue

    # Skip files too large for GitHub
    if src_file.suffix.lower() in SKIP_EXTENSIONS:
        size_mb = src_file.stat().st_size / 1_048_576
        print(f"  ⏭  Skipped large file ({size_mb:.1f} MB): {src_file.relative_to(src)}")
        skipped_large += 1
        continue

    # Also skip individual files over 90 MB
    size_mb = src_file.stat().st_size / 1_048_576
    if size_mb > 90:
        print(f"  ⏭  Skipped oversized file ({size_mb:.1f} MB): {src_file.relative_to(src)}")
        skipped_large += 1
        continue

    # Mirror the path structure inside the repo
    relative  = src_file.relative_to(src)
    dest_file = dest / relative
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_file, dest_file)
    copied += 1

print(f"\n✅ Copied  : {copied} files")
print(f"⏭  Skipped : {skipped_large} large/binary files")
print("   (Large model/data files must be tracked with Git LFS or stored externally)")

## Step 6 — Verify every notebook in the repo passes GitHub's requirements

In [ ]:
repo_notebooks = list(pathlib.Path(LOCAL_REPO).rglob("*.ipynb"))
print(f"Verifying {len(repo_notebooks)} notebook(s) in repo...\n")

all_ok = True

for nb_path in repo_notebooks:
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
        issues = []

        if nb.get("nbformat_minor", 0) < 5:
            issues.append(f"nbformat_minor={nb.get('nbformat_minor')} (needs 5)")
        if not nb.get("metadata", {}).get("kernelspec", {}).get("name"):
            issues.append("missing kernelspec.name")
        if not nb.get("metadata", {}).get("language_info", {}).get("version"):
            issues.append("missing language_info.version")

        # Check all cells have ids
        missing_ids = sum(1 for c in nb.get("cells", []) if not c.get("id"))
        if missing_ids:
            issues.append(f"{missing_ids} cells missing id")

        rel = nb_path.relative_to(LOCAL_REPO)
        if issues:
            all_ok = False
            print(f"  ⚠ {rel}: {', '.join(issues)}")
        else:
            print(f"  ✅ {rel}")

    except Exception as e:
        all_ok = False
        print(f"  ❌ {nb_path.name}: {e}")

print()
if all_ok:
    if repo_notebooks:
        print("✅ All notebooks verified — safe to push!")
    else:
        print("ℹ No notebooks found — pushing other file types only.")
else:
    print("❌ STOP — fix the issues above before pushing.")
    print("   This usually means a notebook was in a subfolder that Step 2 didn't reach.")
    print("   Re-run Step 2 or run the inline fix below:")
    print()
    print("   # Emergency fix for any remaining bad notebooks:")
    print("   for nb_path in pathlib.Path(LOCAL_REPO).rglob('*.ipynb'):")
    print("       nb = json.loads(nb_path.read_text())")
    print("       nb['nbformat_minor'] = 5")
    print("       nb.setdefault('metadata', {})['kernelspec'] = KERNELSPEC")
    print("       nb['metadata']['language_info'] = LANGUAGE_INFO")
    print("       nb_path.write_text(json.dumps(nb, indent=2))")

## Step 7 — Add a README (creates one if it doesn't exist)

In [ ]:
readme_path = pathlib.Path(LOCAL_REPO) / "README.md"

if not readme_path.exists():
    readme_content = f"""# {GITHUB_REPO}

**Researcher:** O. Brown | National University
**Project:** Beyond Threshold Monitoring

---

## Contents

This repository contains Jupyter notebooks and supporting files from the `Merge_Docs_Pro_Folder` project.

| File type | Description |
|-----------|-------------|
| `*.ipynb` | Jupyter notebooks |
| `*.py`    | Python scripts |
| `*.csv`   | Data files |

---
*Large binary/model files (>90 MB) are excluded due to GitHub size limits.*
"""
    readme_path.write_text(readme_content, encoding="utf-8")
    print("✅ README.md created")
else:
    print("✅ README.md already exists — keeping it")

## Step 8 — Commit and push to GitHub

⚠️ **Only run after Step 6 shows all ✅**

In [ ]:
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=LOCAL_REPO)

# Stage everything
subprocess.run(["git", "add", "-A"], cwd=LOCAL_REPO)

# Show summary of what's staged
status = subprocess.run(
    ["git", "diff", "--cached", "--stat"],
    capture_output=True, text=True, cwd=LOCAL_REPO
)
print("Staged changes:")
print(status.stdout or "  (nothing new staged — may already be up to date)")

# Commit
commit = subprocess.run(
    ["git", "commit", "-m",
     "upload: Merge_Docs_Pro_Folder with fixed notebook metadata for GitHub rendering"],
    capture_output=True, text=True, cwd=LOCAL_REPO
)
print(commit.stdout or commit.stderr)

# Push (force to overwrite any previous broken state)
push = subprocess.run(
    ["git", "push", "--force", "origin", "main"],
    capture_output=True, text=True, cwd=LOCAL_REPO
)
print(push.stdout or push.stderr)

if push.returncode == 0:
    print(f"\n✅ SUCCESS! Pushed to GitHub.")
    print(f"   View repo : https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}")
    print("\n⏳ Wait 60 seconds then refresh GitHub — notebooks should render correctly.")
else:
    print("\n❌ Push failed. Check the error above.")
    print("   Common fixes:")
    print("   1. PAT expired → github.com/settings/tokens → regenerate with 'repo' scope")
    print("   2. Update Colab secret: 🔑 sidebar → find Beyond-Threshold-Monitoring → paste new token")
    print("   3. Repo doesn't exist yet → create it at github.com/new (same name, no README)")
    print("   4. Re-run from Step 1")

---
## 🔑 PAT Checklist — if push keeps failing

| Step | Where |
|------|-------|
| Generate/regenerate token | github.com → Settings → Developer Settings → Personal Access Tokens → Tokens (classic) |
| Required scope | ✅ `repo` (full control of private repositories) |
| Save in Colab | 🔑 sidebar → Secrets → `Beyond-Threshold-Monitoring` → paste token → toggle "Notebook access" ON |
| Re-run notebook | From Step 1 |

---
## ✅ What was fixed in every notebook

| Field | Before | After |
|-------|--------|-------|
| `nbformat_minor` | `0` or missing | `5` |
| `metadata.kernelspec` | missing | `{name: python3, ...}` |
| `metadata.language_info.version` | missing | `3.10.12` |
| `cells[*].id` | missing | 8-char hex |
| Cell outputs | large base64 blobs | cleared (code intact) |